# Zero-Shot vs Few-Shot Evaluation

In this notebook, we compare the performance of Zero-Shot and Few-Shot prompting for Yelp review classification.

In [22]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from tqdm import tqdm

from src.config import (
    PREDICTIONS_DIR,
    PROMPT_SUBSET_FILE,
    ZERO_SHOT_PRED_FILE,
    FEW_SHOT_PRED_FILE,
)

from src.prompts import build_zero_shot_prompt, build_few_shot_prompt
from src.llm_runner import call_ollama, parse_prediction, DEFAULT_MODEL
from src.evaluation import compute_metrics, print_metrics

In [23]:
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
print("Predictions dir ready:", PREDICTIONS_DIR)

Predictions dir ready: /home/rehan/Documents/yelp-ai-assignment/outputs/predictions


In [24]:
df = pd.read_csv(PROMPT_SUBSET_FILE)
print("Subset shape:", df.shape)
df.head()

Subset shape: (500, 5)


,text,label_raw,stars,char_length,word_length
0,My family and I tried this for the first time ...,3,4,327,63
1,"If I were playing the word association game, a...",0,1,1809,334
2,I think this would have been a five star if we...,3,4,298,55
3,I prefer the El Hefe on Mill for having more s...,1,2,762,135
4,I just moved into the neighborhood and was try...,1,2,687,129


In [25]:
sample_df = df.head(5).copy()
sample_df

,text,label_raw,stars,char_length,word_length
0,My family and I tried this for the first time ...,3,4,327,63
1,"If I were playing the word association game, a...",0,1,1809,334
2,I think this would have been a five star if we...,3,4,298,55
3,I prefer the El Hefe on Mill for having more s...,1,2,762,135
4,I just moved into the neighborhood and was try...,1,2,687,129


In [26]:
def run_zero_shot_inference(input_df, model_name=DEFAULT_MODEL):
    rows = []

    for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
        review_text = row["text"]
        true_stars = int(row["stars"])

        prompt = build_zero_shot_prompt(review_text)
        raw_output = call_ollama(prompt=prompt, model=model_name, temperature=0.0)
        parsed = parse_prediction(raw_output)

        rows.append({
            "text": review_text,
            "stars": true_stars,
            "prompt_type": "zero_shot",
            **parsed
        })

    return pd.DataFrame(rows)

In [27]:
def run_few_shot_inference(input_df, model_name=DEFAULT_MODEL):
    rows = []

    for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
        review_text = row["text"]
        true_stars = int(row["stars"])

        prompt = build_few_shot_prompt(review_text)
        raw_output = call_ollama(prompt=prompt, model=model_name, temperature=0.0)
        parsed = parse_prediction(raw_output)

        rows.append({
            "text": review_text,
            "stars": true_stars,
            "prompt_type": "few_shot",
            **parsed
        })

    return pd.DataFrame(rows)

In [28]:
test_prompt = """
Return only valid JSON:
{"stars": 5, "explanation": "Very positive review."}
"""

raw = call_ollama(test_prompt, model=DEFAULT_MODEL)
print(raw)
print(parse_prediction(raw))

{"stars": 5, "explanation": "Very positive review."}
{'raw_output': '{"stars": 5, "explanation": "Very positive review."}', 'json_valid': True, 'stars_pred': 5, 'explanation_pred': 'Very positive review.', 'parse_error': None}


In [29]:
df = pd.read_csv(PROMPT_SUBSET_FILE)
print(df.shape)
df.head()

(500, 5)


,text,label_raw,stars,char_length,word_length
0,My family and I tried this for the first time ...,3,4,327,63
1,"If I were playing the word association game, a...",0,1,1809,334
2,I think this would have been a five star if we...,3,4,298,55
3,I prefer the El Hefe on Mill for having more s...,1,2,762,135
4,I just moved into the neighborhood and was try...,1,2,687,129


In [30]:
sample_df = df.head(5).copy()
sample_df[["text", "stars"]]

,text,stars
0,My family and I tried this for the first time ...,4
1,"If I were playing the word association game, a...",1
2,I think this would have been a five star if we...,4
3,I prefer the El Hefe on Mill for having more s...,2
4,I just moved into the neighborhood and was try...,2


In [31]:
zero_test = run_zero_shot_inference(sample_df, model_name=DEFAULT_MODEL)
zero_test[["stars", "stars_pred", "json_valid", "parse_error", "explanation_pred"]]

100%|███████████████████| 5/5 [00:06<00:00,  1.36s/it]


,stars,stars_pred,json_valid,parse_error,explanation_pred
0,4,4,True,None,Generally positive experience with unique food...
1,1,2,True,None,The reviewer expresses disappointment and a la...
2,4,3,True,None,Mixed experience with ambiance and drinks exce...
3,2,2,True,None,Reviewer expresses strong dissatisfaction with...
4,2,1,True,None,Reviewer had a very negative experience with t...


In [32]:
few_test = run_few_shot_inference(sample_df, model_name=DEFAULT_MODEL)
few_test[["stars", "stars_pred", "json_valid", "parse_error", "explanation_pred"]]

100%|███████████████████| 5/5 [00:07<00:00,  1.47s/it]


,stars,stars_pred,json_valid,parse_error,explanation_pred
0,4,4,True,None,Generally positive review with praise for uniq...
1,1,2,True,None,The reviewer expresses disappointment and a la...
2,4,3,True,None,Mixed review with praise for ambiance and drin...
3,2,2,True,None,"Negative review citing poor service, pretentio..."
4,2,1,True,None,"Negative review citing unfriendly reception, j..."


In [33]:
zero_metrics_small = compute_metrics(zero_test)
few_metrics_small = compute_metrics(few_test)

print_metrics("ZERO-SHOT SMALL TEST", zero_metrics_small)
print_metrics("FEW-SHOT SMALL TEST", few_metrics_small)


=== ZERO-SHOT SMALL TEST ===
total_rows: 5
json_compliance_rate: 1.0
valid_prediction_rows: 5
accuracy: 0.4
macro_f1: 0.2917

=== FEW-SHOT SMALL TEST ===
total_rows: 5
json_compliance_rate: 1.0
valid_prediction_rows: 5
accuracy: 0.4
macro_f1: 0.2917


In [34]:
mini_df = df.head(50).copy()

zero_mini = run_zero_shot_inference(mini_df, model_name=DEFAULT_MODEL)
few_mini = run_few_shot_inference(mini_df, model_name=DEFAULT_MODEL)

zero_mini_metrics = compute_metrics(zero_mini)
few_mini_metrics = compute_metrics(few_mini)

print_metrics("ZERO-SHOT 50 ROWS", zero_mini_metrics)
print_metrics("FEW-SHOT 50 ROWS", few_mini_metrics)

100%|█████████████████| 50/50 [01:07<00:00,  1.34s/it]


=== ZERO-SHOT 50 ROWS ===
total_rows: 50
json_compliance_rate: 1.0
valid_prediction_rows: 50
accuracy: 0.7
macro_f1: 0.5959

=== FEW-SHOT 50 ROWS ===
total_rows: 50
json_compliance_rate: 1.0
valid_prediction_rows: 50
accuracy: 0.72
macro_f1: 0.6125


In [35]:
print("Zero invalid JSON:", (zero_mini["json_valid"] == False).sum())
print("Few invalid JSON :", (few_mini["json_valid"] == False).sum())

Zero invalid JSON: 0
Few invalid JSON : 0


In [36]:
zero_mini_failures = zero_mini[
    (zero_mini["json_valid"] == False) |
    (zero_mini["stars"] != zero_mini["stars_pred"])
]

few_mini_failures = few_mini[
    (few_mini["json_valid"] == False) |
    (few_mini["stars"] != few_mini["stars_pred"])
]

zero_mini_failures[["text", "stars", "stars_pred", "parse_error", "explanation_pred"]].head(10)

,text,stars,stars_pred,parse_error,explanation_pred
1,"If I were playing the word association game, a...",1,2,None,The reviewer expresses disappointment and a la...
2,I think this would have been a five star if we...,4,3,None,Mixed experience with ambiance and drinks exce...
4,I just moved into the neighborhood and was try...,2,1,None,Reviewer had a very negative experience with u...
5,"Went for dim sum. It was moderately busy, but...",4,3,None,The reviewer found the experience to be averag...
6,I've been here now about 5-6 times now and am ...,4,5,None,The reviewer enthusiastically praises the rest...
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,None,Mixed sentiment with praise for creativity and...
11,This was a great disappointment. My boyfriend ...,2,1,None,"Reviewer's experience was largely negative, ci..."
16,i'm sooo glad we didn't have a lot time in her...,5,4,None,"Praise for the store's appearance, helpful sta..."
24,While perusing I saw this Chipotle and thought...,1,2,None,Mixed experience with high expectations from r...
27,Located in a new shopping center in NW Phoenix...,3,4,None,"Positive experience with good food, fast and f..."


In [37]:
few_mini_failures[["text", "stars", "stars_pred", "parse_error", "explanation_pred"]].head(10)

,text,stars,stars_pred,parse_error,explanation_pred
1,"If I were playing the word association game, a...",1,2,None,The reviewer expresses disappointment and a la...
2,I think this would have been a five star if we...,4,3,None,Mixed review with praise for ambiance and drin...
4,I just moved into the neighborhood and was try...,2,1,None,"Negative review citing unfriendly reception, j..."
5,"Went for dim sum. It was moderately busy, but...",4,3,None,Generally positive review with praise for food...
6,I've been here now about 5-6 times now and am ...,4,5,None,Overwhelmingly positive review with praise for...
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,None,Mixed sentiment with mild praise for creativit...
11,This was a great disappointment. My boyfriend ...,2,1,None,"Strongly negative review with disappointment, ..."
16,i'm sooo glad we didn't have a lot time in her...,5,3,None,Mixed sentiment with praise for the store's of...
24,While perusing I saw this Chipotle and thought...,1,2,None,Mixed experience with high praise for food qua...
27,Located in a new shopping center in NW Phoenix...,3,4,None,"Positive review with praise for food, staff, a..."


In [38]:
zero_shot_df = run_zero_shot_inference(df, model_name=DEFAULT_MODEL)
few_shot_df = run_few_shot_inference(df, model_name=DEFAULT_MODEL)

zero_metrics = compute_metrics(zero_shot_df)
few_metrics = compute_metrics(few_shot_df)

print_metrics("ZERO-SHOT FULL", zero_metrics)
print_metrics("FEW-SHOT FULL", few_metrics)

100%|███████████████| 500/500 [10:39<00:00,  1.28s/it]


=== ZERO-SHOT FULL ===
total_rows: 500
json_compliance_rate: 1.0
valid_prediction_rows: 500
accuracy: 0.708
macro_f1: 0.6993

=== FEW-SHOT FULL ===
total_rows: 500
json_compliance_rate: 1.0
valid_prediction_rows: 500
accuracy: 0.712
macro_f1: 0.706


In [39]:
zero_shot_df.to_csv(ZERO_SHOT_PRED_FILE, index=False)
few_shot_df.to_csv(FEW_SHOT_PRED_FILE, index=False)

In [40]:
#Zero-shot full
few_shot_df = run_few_shot_inference(df, model_name=DEFAULT_MODEL)
few_shot_df.to_csv(FEW_SHOT_PRED_FILE, index=False)
print_metrics("FEW-SHOT FULL", compute_metrics(few_shot_df))

100%|███████████████| 500/500 [10:47<00:00,  1.29s/it]


=== FEW-SHOT FULL ===
total_rows: 500
json_compliance_rate: 1.0
valid_prediction_rows: 500
accuracy: 0.712
macro_f1: 0.706


In [41]:
#Few-shot full
few_shot_df = run_few_shot_inference(df, model_name=DEFAULT_MODEL)
few_shot_df.to_csv(FEW_SHOT_PRED_FILE, index=False)
print_metrics("FEW-SHOT FULL", compute_metrics(few_shot_df))

100%|███████████████| 500/500 [10:20<00:00,  1.24s/it]


=== FEW-SHOT FULL ===
total_rows: 500
json_compliance_rate: 1.0
valid_prediction_rows: 500
accuracy: 0.712
macro_f1: 0.706


In [42]:
zero_metrics = compute_metrics(zero_shot_df)
few_metrics = compute_metrics(few_shot_df)

comparison_df = pd.DataFrame([
    {"method": "zero_shot", **zero_metrics},
    {"method": "few_shot", **few_metrics},
])

comparison_df

,method,total_rows,json_compliance_rate,valid_prediction_rows,accuracy,macro_f1
0,zero_shot,500,1.0,500,0.708,0.6993
1,few_shot,500,1.0,500,0.712,0.7060


In [43]:
#Failure analysis
zero_failures = zero_shot_df[
    (zero_shot_df["json_valid"] == False) |
    (zero_shot_df["stars"] != zero_shot_df["stars_pred"])
].copy()

few_failures = few_shot_df[
    (few_shot_df["json_valid"] == False) |
    (few_shot_df["stars"] != few_shot_df["stars_pred"])
].copy()

print("Zero-shot failures:", len(zero_failures))
print("Few-shot failures :", len(few_failures))

Zero-shot failures: 146
Few-shot failures : 144


In [44]:
zero_failures[["text", "stars", "stars_pred", "explanation_pred", "parse_error"]].head(10)
few_failures[["text", "stars", "stars_pred", "explanation_pred", "parse_error"]].head(10)

,text,stars,stars_pred,explanation_pred,parse_error
1,"If I were playing the word association game, a...",1,2,The reviewer expresses disappointment and a la...,None
2,I think this would have been a five star if we...,4,3,Mixed review with praise for ambiance and drin...,None
4,I just moved into the neighborhood and was try...,2,1,"Negative review citing unfriendly reception, j...",None
5,"Went for dim sum. It was moderately busy, but...",4,3,Generally positive review with praise for food...,None
6,I've been here now about 5-6 times now and am ...,4,5,Overwhelmingly positive review with praise for...,None
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,Mixed sentiment with mild praise for creativit...,None
11,This was a great disappointment. My boyfriend ...,2,1,"Strongly negative review with disappointment, ...",None
16,i'm sooo glad we didn't have a lot time in her...,5,3,Mixed sentiment with praise for the store's of...,None
24,While perusing I saw this Chipotle and thought...,1,2,Mixed experience with high praise for food qua...,None
27,Located in a new shopping center in NW Phoenix...,3,4,"Positive review with praise for food, staff, a...",None
